In [1]:
import os
import json
import pandas as pd
from tqdm.notebook import tqdm


In [2]:
# Update this path to your dataset location
DATASET_PATH = "/Users/ashishjain/Documents/assignment/transcript-intelligence/data/raw"


# Check how many folders exist
folders = [
    f for f in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, f))
]

print(f"✅ Found {len(folders)} call folders")

✅ Found 100 call folders


In [3]:
# Test with first folder
sample_folder = os.path.join(DATASET_PATH, folders[0])
print(folders[0])

# Load each file
with open(f"{sample_folder}/meeting-info.json") as f:
    meeting = json.load(f)

with open(f"{sample_folder}/summary.json") as f:
    summary = json.load(f)

with open(f"{sample_folder}/transcript.json") as f:
    transcript = json.load(f)

# Preview
print("📋 MEETING INFO:")
print(f"   Title    : {meeting['title']}")
print(f"   Duration : {meeting['duration']} mins")
print(f"   Start    : {meeting['startTime']}")
print(f"   Emails   : {meeting['allEmails']}")
print("\n📊 SUMMARY:")
print(f"   Sentiment : {summary['overallSentiment']}")
print(f"   Score     : {summary['sentimentScore']}")
print(f"   Topics    : {summary['topics']}")
print(f"   Actions   : {len(summary['actionItems'])} items")

print("\n💬 TRANSCRIPT TYPE:", type(transcript))
print("   Preview   :", str(transcript)[:200])

01KQ7F58A2E07B857C24624D
📋 MEETING INFO:
   Title    : Competitive Landscape Review
   Duration : 20.2 mins
   Start    : 2026-02-19T09:15:00.000Z
   Emails   : ['jordan.whitfield@aegiscloud.com', 'tom.bradley@aegiscloud.com']

📊 SUMMARY:
   Sentiment : mixed-positive
   Score     : 3.4
   Topics    : ['competitive analysis', 'product gaps', 'siem integration', 'compliance features', 'pricing pressure', 'roadmap prioritization']
   Actions   : 4 items

💬 TRANSCRIPT TYPE: <class 'dict'>
   Preview   : {'data': [{'sentence': 'Hey Tom, you there? I think you might be on mute.', 'speaker_name': 'Jordan Whitfield', 'sentimentType': 'neutral', 'speaker_id': 0, 'time': 4.6, 'endTime': 8.9, 'averageConfid


In [4]:
# Look at first 3 sentences
sentences = transcript['data']

print(f"📝 Total sentences: {len(sentences)}")
print(f"\n🗣️ First 3 sentences:")
for s in sentences[:3]:
    print(f"   [{s['speaker_name']}] ({s['sentimentType']}) → {s['sentence']}")
    
print(f"\n🎭 Unique sentiment types in this call:")
sentiments = set(s['sentimentType'] for s in sentences)
print(f"   {sentiments}")

print(f"\n👥 Unique speakers:")
speakers = set(s['speaker_name'] for s in sentences)
print(f"   {speakers}")

📝 Total sentences: 36

🗣️ First 3 sentences:
   [Jordan Whitfield] (neutral) → Hey Tom, you there? I think you might be on mute.
   [Tom Bradley] (neutral) → Oh, yeah sorry, I was — yeah, I had myself muted. Can you hear me now?
   [Jordan Whitfield] (neutral) → Yeah, loud and clear. Okay cool, so I figured we should probably carve out some time to actually go through the competitive landscape before the product review next week, because I feel like we've been kind of operating on vibes a little bit.

🎭 Unique sentiment types in this call:
   {'negative', 'positive', 'neutral'}

👥 Unique speakers:
   {'Tom Bradley', 'Jordan Whitfield'}


In [5]:
def load_all_calls(dataset_path):
    folders = [
        f for f in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, f))
    ]

    data = []
    errors = []

    for folder in tqdm(folders, desc="Loading calls"):
        folder_path = os.path.join(dataset_path, folder)

        try:
            # Load files
            with open(f"{folder_path}/meeting-info.json") as f:
                meeting = json.load(f)
            with open(f"{folder_path}/summary.json") as f:
                summary = json.load(f)
            with open(f"{folder_path}/transcript.json") as f:
                transcript = json.load(f)

            # Extract sentences
            sentences = transcript.get("data", [])
            full_text = " ".join([s["sentence"] for s in sentences])

            # Sentence-level sentiment counts
            sent_counts = {"positive": 0, "negative": 0, "neutral": 0}
            for s in sentences:
                t = s.get("sentimentType", "neutral")
                if t in sent_counts:
                    sent_counts[t] += 1

            record = {
                # Meeting Info
                "meeting_id"        : meeting.get("meetingId"),
                "title"             : meeting.get("title"),
                "duration_mins"     : meeting.get("duration"),
                "start_time"        : meeting.get("startTime"),
                "participants"      : meeting.get("allEmails", []),
                "num_participants"  : len(meeting.get("allEmails", [])),

                # Summary
                "summary"           : summary.get("summary"),
                "topics"            : summary.get("topics", []),
                "sentiment"         : summary.get("overallSentiment"),
                "sentiment_score"   : summary.get("sentimentScore"),
                "action_items"      : summary.get("actionItems", []),
                "num_action_items"  : len(summary.get("actionItems", [])),
                "key_moments"       : summary.get("keyMoments", []),

                # Transcript
                "full_text"         : full_text,
                "num_sentences"     : len(sentences),
                "positive_sents"    : sent_counts["positive"],
                "negative_sents"    : sent_counts["negative"],
                "neutral_sents"     : sent_counts["neutral"],
                "speakers"          : list(set(s["speaker_name"] for s in sentences)),
                "num_speakers"      : len(set(s["speaker_name"] for s in sentences)),
            }

            data.append(record)

        except Exception as e:
            errors.append({"folder": folder, "error": str(e)})

    print(f"\n✅ Loaded    : {len(data)} calls")
    print(f"❌ Errors    : {len(errors)} calls")
    if errors:
        for e in errors:
            print(f"   {e['folder']} → {e['error']}")

    return pd.DataFrame(data)


# Run it
df = load_all_calls(DATASET_PATH)
print(f"\n📊 DataFrame shape: {df.shape}")
print(f"📋 Columns: {df.columns.tolist()}")

Loading calls:   0%|          | 0/100 [00:00<?, ?it/s]


✅ Loaded    : 100 calls
❌ Errors    : 0 calls

📊 DataFrame shape: (100, 20)
📋 Columns: ['meeting_id', 'title', 'duration_mins', 'start_time', 'participants', 'num_participants', 'summary', 'topics', 'sentiment', 'sentiment_score', 'action_items', 'num_action_items', 'key_moments', 'full_text', 'num_sentences', 'positive_sents', 'negative_sents', 'neutral_sents', 'speakers', 'num_speakers']


In [6]:
##  Detect the call type 

def classify_call_type(title):
    title_lower = title.lower()

    if any(word in title_lower for word in [
        "support", "case #", "issue", "ticket",
        "billing", "incident", "help"
    ]):
        return "Customer Support"

    elif any(word in title_lower for word in [
        "sync", "standup", "stand-up", "internal",
        "planning", "escalation", "retrospective",
        "review", "team", "engineering", "debrief"
    ]):
        return "Internal"

    else:
        return "External"


df["call_type"] = df["title"].apply(classify_call_type)

print("📞 Call Type Distribution:")
print(df["call_type"].value_counts())
print()
print("📊 Sentiment Distribution:")
print(df["sentiment"].value_counts())
print()
print("⏱️ Avg Duration by Call Type:")
print(df.groupby("call_type")["duration_mins"].mean().round(1))

📞 Call Type Distribution:
call_type
External            39
Internal            31
Customer Support    30
Name: count, dtype: int64

📊 Sentiment Distribution:
sentiment
mixed-positive    33
mixed-negative    33
very-positive     21
positive           7
negative           4
very-negative      2
Name: count, dtype: int64

⏱️ Avg Duration by Call Type:
call_type
Customer Support    19.3
External            35.9
Internal            34.0
Name: duration_mins, dtype: float64


In [7]:
# Save to CSV for use in other notebooks
df.to_csv("/Users/ashishjain/Documents/assignment/transcript-intelligence/data/processed/all_calls.csv", index=False)
print("✅ Saved → data/processed/all_calls.csv")

# Quick preview
df[["title", "call_type", "sentiment",
    "sentiment_score", "duration_mins",
    "num_sentences", "num_action_items"]].head(10)

✅ Saved → data/processed/all_calls.csv


,title,call_type,sentiment,sentiment_score,duration_mins,num_sentences,num_action_items
0,Competitive Landscape Review,Internal,mixed-positive,3.4,20.2,36,4
1,Detect Outage - Root Cause Analysis,External,mixed-negative,2.4,15.9,38,4
2,Aegis / Summit Trust - Platform Concerns Discu...,External,mixed-negative,2.4,53.0,47,4
3,Support Case #6977 - Brightpath Commerce Slow ...,Customer Support,mixed-negative,2.8,15.8,35,4
4,Aegis / Brightpath Commerce - Detect Module De...,External,mixed-positive,3.9,45.8,44,4
5,Detect Outage - Escalation Bridge,Internal,negative,1.8,17.1,41,4
6,Aegis / Trailhead Marketplace - Renewal Confir...,External,positive,4.5,46.6,49,4
7,ESCALATION: Northstar Pharma - Detect Outage I...,Internal,mixed-negative,2.1,13.3,41,4
8,Support Case #3536 - Stratos Cloud LogVault In...,Customer Support,mixed-negative,2.4,12.8,41,3
9,Aegis / Cobalt Software - Q2 Planning,Internal,mixed-positive,3.4,41.2,42,4


In [8]:
# Check actual column names
print(df.columns.tolist())
print(f"\nShape: {df.shape}")

['meeting_id', 'title', 'duration_mins', 'start_time', 'participants', 'num_participants', 'summary', 'topics', 'sentiment', 'sentiment_score', 'action_items', 'num_action_items', 'key_moments', 'full_text', 'num_sentences', 'positive_sents', 'negative_sents', 'neutral_sents', 'speakers', 'num_speakers', 'call_type']

Shape: (100, 21)
